In [1]:
import sqlite3
import numpy as np
import pandas as pd

# Connect to the local SQLite database
conn = sqlite3.connect("data/data.db")

# Load data and rename columns to match HW2 instructions
query = """
SELECT 
    date AS ts,
    symbol AS ticker,
    open,
    high,
    low,
    close,
    adj_close,
    volume
FROM signals
"""

df_raw = pd.read_sql_query(query, conn)
df_raw.head()

,ts,ticker,open,high,low,close,adj_close,volume
0,2015-01-07,NWE,56.05,56.18,55.36,56.10,38.01,403034
1,2015-01-05,OTTR,30.74,31.06,30.61,30.85,22.82,102305
2,2015-01-05,TMP,53.96,54.55,52.39,53.27,39.68,51510
3,2015-01-05,PKOH,60.68,61.86,59.52,59.76,50.24,34422
4,2015-01-05,MRIN,357.73,361.93,354.79,355.21,355.21,2065


In [2]:
# Normalize tickers and convert timestamps
df_raw["ticker"] = df_raw["ticker"].astype(str).str.strip().str.upper()
df_raw["ts"] = pd.to_datetime(df_raw["ts"])

# Display structure and descriptive statistics
print("--- Data Info ---")
df_raw.info()

print("\n--- Descriptive Statistics ---")
display(df_raw.describe())

# Extract date range and unique tickers
start_date = df_raw["ts"].min().strftime("%Y-%m-%d")
end_date = df_raw["ts"].max().strftime("%Y-%m-%d")
tickers = sorted(df_raw["ticker"].unique())

print(f"\nDate range: {start_date} to {end_date}")
print(f"Unique tickers ({len(tickers)}): {tickers[:10]}...")

--- Data Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9697099 entries, 0 to 9697098
Data columns (total 8 columns):
 #   Column     Dtype         
---  ------     -----         
 0   ts         datetime64[ns]
 1   ticker     object        
 2   open       float64       
 3   high       float64       
 4   low        float64       
 5   close      float64       
 6   adj_close  float64       
 7   volume     int64         
dtypes: datetime64[ns](1), float64(5), int64(1), object(1)
memory usage: 591.9+ MB

--- Descriptive Statistics ---


,ts,open,high,low,close,adj_close,volume
count,9697099,9.697099e+06,9.697099e+06,9.697099e+06,9.697099e+06,9.697099e+06,9.697099e+06
mean,2020-12-09 08:19:21.685156864,5.683236e+01,5.783944e+01,5.585148e+01,5.679607e+01,5.413932e+01,1.732416e+06
min,2015-01-02 00:00:00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e-06,0.000000e+00,0.000000e+00
25%,2018-05-18 00:00:00,1.060000e+01,1.084890e+01,1.037000e+01,1.060000e+01,9.730000e+00,6.802600e+04
50%,2021-03-24 00:00:00,2.493000e+01,2.527000e+01,2.454000e+01,2.492000e+01,2.253000e+01,3.232000e+05
75%,2023-08-01 00:00:00,5.580000e+01,5.663000e+01,5.496000e+01,5.579000e+01,5.156000e+01,1.138886e+06
max,2025-12-31 00:00:00,4.111508e+05,5.597119e+05,3.467624e+05,3.485609e+05,3.485609e+05,5.747791e+09
std,NaN,2.653002e+02,3.062758e+02,2.452368e+02,2.536716e+02,2.526424e+02,1.214516e+07



Date range: 2015-01-02 to 2025-12-31
Unique tickers (5417): ['A', 'AA', 'AADI', 'AAIC', 'AAIN', 'AAL', 'AAM', 'AAMC', 'AAME', 'AAMI']...


In [3]:
# Cell 3: Detect Data Quality Issues - Prices, Volumes, Returns
import warnings
warnings.filterwarnings('ignore')

print("Sorting data and removing exact duplicates...")
df = df_raw.copy()
# Sort by ticker and date before calculating returns
df = df.sort_values(by=["ticker", "ts"]).reset_index(drop=True)
df = df.drop_duplicates()

issues_list = []

# Helper function to append issues to the report
def log_issues(mask, issue_type, desc, severity):
    invalid = df[mask]
    if not invalid.empty:
        res = pd.DataFrame({
            "ticker": invalid["ticker"],
            "issue_type": issue_type,
            "issue_date": invalid["ts"],
            "description": desc,
            "severity": severity
        })
        issues_list.append(res)

print("Checking for missing values in columns...")
for col in ["open", "high", "low", "close", "adj_close", "volume"]:
    log_issues(df[col].isna(), "Missing Value", f"Missing value in {col}", "medium")

print("Checking for non-positive or non-finite prices...")
for col in ["open", "high", "low", "close", "adj_close"]:
    mask = (df[col] <= 0) | (~np.isfinite(df[col]))
    log_issues(mask, "Invalid Price", f"{col} is non-positive or non-finite", "high")

print("Checking OHLC bounds (e.g., high < close)...")
mask_bounds = (df["low"] > df["high"]) | (df["low"] > df["open"]) | (df["open"] > df["high"]) | (df["low"] > df["close"]) | (df["close"] > df["high"])
log_issues(mask_bounds, "Price Inconsistency", "OHLC bounds violated", "high")

print("Checking for zero volume...")
log_issues(df["volume"] == 0, "Zero Volume", "Volume is exactly 0", "low")

print("Calculating daily returns and checking for outliers (>3 std dev)...")
# Calculate daily returns: (close - prev_close) / prev_close
df["prev_close"] = df.groupby("ticker")["close"].shift(1)
df["return"] = (df["close"] - df["prev_close"]) / df["prev_close"]

# Calculate mean and sample std dev (ddof=1) per ticker
stats = df.groupby("ticker")["return"].agg(mean_ret="mean", std_ret="std")
df = df.merge(stats, on="ticker", how="left")

# Flag outliers (excluding undefined/0 std dev)
valid_std = (df["std_ret"] > 0) & (df["std_ret"].notna())
mask_outliers = valid_std & (np.abs(df["return"] - df["mean_ret"]) > 3 * df["std_ret"])
log_issues(mask_outliers, "Extreme Return", "Return exceeds 3 standard deviations", "medium")

# Compile the report
if issues_list:
    dq_report_prices = pd.concat(issues_list, ignore_index=True)
else:
    dq_report_prices = pd.DataFrame(columns=["ticker", "issue_type", "issue_date", "description", "severity"])

print(f"\nDone! Found {len(dq_report_prices)} issues related to prices, volume, and returns.")
display(dq_report_prices.head())

Sorting data and removing exact duplicates...
Checking for missing values in columns...
Checking for non-positive or non-finite prices...
Checking OHLC bounds (e.g., high < close)...
Checking for zero volume...
Calculating daily returns and checking for outliers (>3 std dev)...

Done! Found 138715 issues related to prices, volume, and returns.


,ticker,issue_type,issue_date,description,severity
0,PTVE,Invalid Price,2025-04-01,open is non-positive or non-finite,high
1,VERV,Invalid Price,2025-07-24,open is non-positive or non-finite,high
2,VOXX,Invalid Price,2025-04-01,open is non-positive or non-finite,high
3,PTVE,Invalid Price,2025-04-01,high is non-positive or non-finite,high
4,VOXX,Invalid Price,2025-04-01,high is non-positive or non-finite,high


In [4]:
# Cell 4: Detect Missing Trading Days and Finalize Report
print("Extracting global trading calendar...")
global_dates = pd.Series(np.sort(df["ts"].unique()))

missing_issues = []

print("Checking for missing dates per ticker (this may take ~10-20 seconds)...")
for ticker, group in df.groupby("ticker"):
    t_min, t_max = group["ts"].min(), group["ts"].max()
    
    # Expected dates based on global calendar within ticker's active lifespan
    expected = global_dates[(global_dates >= t_min) & (global_dates <= t_max)]
    actual = group["ts"].values
    
    # Identify gaps
    missing = expected[~expected.isin(actual)]
    
    if not missing.empty:
        missing_issues.append(pd.DataFrame({
            "ticker": ticker,
            "issue_type": "Missing Date",
            "issue_date": missing,
            "description": f"Missing trading day between {t_min.date()} and {t_max.date()}",
            "severity": "medium"
        }))

# Compile missing dates report
if missing_issues:
    dq_report_gaps = pd.concat(missing_issues, ignore_index=True)
else:
    dq_report_gaps = pd.DataFrame(columns=dq_report_prices.columns)

# Finalize the comprehensive Data Quality Report
dq_report = pd.concat([dq_report_prices, dq_report_gaps], ignore_index=True)

print(f"Found {len(dq_report_gaps)} missing date issues.")
print(f"Total issues in Data Quality Report: {len(dq_report)}")

# Display a random sample of the final report to verify variety
display(dq_report.sample(10, random_state=42))

Extracting global trading calendar...
Checking for missing dates per ticker (this may take ~10-20 seconds)...
Found 602435 missing date issues.
Total issues in Data Quality Report: 741150


,ticker,issue_type,issue_date,description,severity
319993,EMP,Missing Date,2016-04-21,Missing trading day between 2015-01-20 and 202...,medium
667671,TCBS,Missing Date,2022-03-14,Missing trading day between 2021-07-16 and 202...,medium
161300,AIC,Missing Date,2019-10-22,Missing trading day between 2015-03-20 and 202...,medium
302603,DTOC,Missing Date,2023-05-23,Missing trading day between 2022-09-30 and 202...,medium
456793,LSBK,Missing Date,2017-01-17,Missing trading day between 2015-01-02 and 202...,medium
670606,TCON,Missing Date,2023-05-22,Missing trading day between 2018-05-16 and 202...,medium
345499,FKWL,Missing Date,2019-10-29,Missing trading day between 2015-01-02 and 202...,medium
369682,GERN,Missing Date,2021-12-20,Missing trading day between 2015-11-05 and 202...,medium
509964,NES,Missing Date,2018-04-27,Missing trading day between 2017-10-12 and 202...,medium
423738,IVR,Missing Date,2022-03-30,Missing trading day between 2015-08-17 and 202...,medium


In [6]:
# Cell 5: Clean Data and Save ohlc_hw_cleaned to SQLite
print("Starting data cleaning pipeline...")

# 1. Work on a copy sorted by ticker and date
df_clean = df_raw.copy().sort_values(by=["ticker", "ts"]).reset_index(drop=True)

# 2. Drop rows where ALL raw price columns are missing
price_cols = ["open", "high", "low", "close"]
df_clean = df_clean.dropna(subset=price_cols, how="all").reset_index(drop=True)

# 3. Identify invalid prices (non-positive or bound violations)
invalid_price_mask = (
    (df_clean["open"] <= 0) | (df_clean["high"] <= 0) | 
    (df_clean["low"] <= 0) | (df_clean["close"] <= 0) |
    (df_clean["low"] > df_clean["high"]) | 
    (df_clean["low"] > df_clean["open"]) | (df_clean["low"] > df_clean["close"]) |
    (df_clean["high"] < df_clean["open"]) | (df_clean["high"] < df_clean["close"])
)

# Audit trail: flag repaired rows
df_clean["is_repaired"] = invalid_price_mask

# Set invalid prices to NaN for forward/backward filling
for col in price_cols:
    df_clean.loc[invalid_price_mask, col] = np.nan

# 4. Forward-fill and backward-fill prices within ticker groups
df_clean[price_cols] = df_clean.groupby("ticker")[price_cols].ffill()
df_clean[price_cols] = df_clean.groupby("ticker")[price_cols].bfill()

# Drop residual rows that could not be repaired (unresolvable bad tickers)
df_clean = df_clean.dropna(subset=price_cols).reset_index(drop=True)
df_clean = df_clean[(df_clean[price_cols] > 0).all(axis=1)].reset_index(drop=True)

# 5. Enforce strict OHLC bounds
df_clean["high"] = df_clean[["open", "high", "low", "close"]].max(axis=1)
df_clean["low"] = df_clean[["open", "high", "low", "close"]].min(axis=1)

# 6. Handle volume: fill NaNs with 0 and add repair flag
df_clean["volume_repaired"] = df_clean["volume"].isna()
df_clean["volume"] = df_clean["volume"].fillna(0).astype(int)

# 7. Handle adjusted close: ffill/bfill within ticker, fallback to close
if "adj_close" in df_clean.columns:
    df_clean["adj_close"] = df_clean.groupby("ticker")["adj_close"].ffill()
    df_clean["adj_close"] = df_clean.groupby("ticker")["adj_close"].bfill()
    df_clean["adj_close"] = df_clean["adj_close"].fillna(df_clean["close"])
    invalid_adj = (df_clean["adj_close"] <= 0) | df_clean["adj_close"].isna()
    df_clean.loc[invalid_adj, "adj_close"] = df_clean.loc[invalid_adj, "close"]

# 8. Store timestamp as ISO date string (YYYY-MM-DD) for SQL compatibility
df_clean["ts"] = pd.to_datetime(df_clean["ts"]).dt.strftime("%Y-%m-%d")

# 9. Validation assertions
all_prices = ["open", "high", "low", "close", "adj_close"]
assert df_clean["ts"].notna().all(), "Null dates found!"
assert (df_clean[all_prices] > 0).all().all(), "Non-positive prices remaining!"
assert (df_clean["low"] <= df_clean["high"]).all(), "Invalid low/high bounds!"
assert not df_clean.duplicated(subset=["ticker", "ts"]).any(), "Duplicate keys found!"

print("Data cleaning completed successfully!")

# 10. Save cleaned table to SQLite database
conn = sqlite3.connect("data/data.db")
df_clean.to_sql("ohlc_hw_cleaned", conn, if_exists="replace", index=False)
conn.close()

print("\nTable 'ohlc_hw_cleaned' successfully saved to data/data.db.")
print(f"Final dataset shape: {df_clean.shape}")
display(df_clean.head())

Starting data cleaning pipeline...
Data cleaning completed successfully!

Table 'ohlc_hw_cleaned' successfully saved to data/data.db.
Final dataset shape: (9697096, 10)


,ts,ticker,open,high,low,close,adj_close,volume,is_repaired,volume_repaired
0,2015-01-02,A,41.18,41.31,40.37,40.56,37.42,1530798,False,False
1,2015-01-05,A,40.32,40.46,39.70,39.80,36.72,2042240,False,False
2,2015-01-06,A,39.81,40.02,39.02,39.18,36.15,2084562,False,False
3,2015-01-07,A,39.52,39.81,39.29,39.70,36.63,3359700,False,False
4,2015-01-08,A,40.24,40.98,40.18,40.89,37.73,2116341,False,False
